# Q1. Change in the proportion of negligent fires

**Research question:** Did the proportion of negligent fires among human-caused fires change between the pre-2017 and post-2017 periods, and was the change the same in the critical summer period and in the non-critical months?

**Test:** Two-proportion Z-test, run separately for each season.

### Hypotheses (for each season)

- H₀: \(p_{pre}=p_{post}\) — the proportion of negligent fires is the same before and after 2017.
- H₁: \(p_{pre}\ne p_{post}\) — the proportion changed after 2017.

Significance level: \(\alpha=0.05\).

Here, \(p_{pre}\) is the pre-2017 proportion and \(p_{post}\) is the post-2017 proportion.

### Data

| Period | Season | Intentional | Negligent | Total | Negligent share |
|---|---|---:|---:|---:|---:|
| Pre-2017 | Critical | 1,120 | 1,411 | 2,531 | 55.7% |
| Post-2017 | Critical | 639 | 576 | 1,215 | 47.4% |
| Pre-2017 | Non-critical | 261 | 1,242 | 1,503 | 82.6% |
| Post-2017 | Non-critical | 150 | 1,120 | 1,270 | 88.2% |

The pre-2017 period covers 16 years and the post-2017 period 8, so raw counts are not directly comparable. The Z-test compares proportions, rather than raw counts.

*These aggregated counts should come from the project's validated human-caused fire records: Cause_Type (Intentional/Negligent), Period (2001–2016/2018–2025), and Season_Risk_Period (critical/non-critical).*

## 1. Enter the observed counts

The number of **negligent fires** is the number of successes. The total number of human-caused fires is the sample size for each period and season.

The counts below are the aggregated results shown in the table above.

In [1]:
import pandas as pd
from scipy.stats import norm

# Aggregated counts from the validated wildfire dataset
data = {
    "Critical": {
        "pre_negligent": 1411,
        "pre_total": 2531,
        "post_negligent": 576,
        "post_total": 1215
    },
    "Non-critical": {
        "pre_negligent": 1242,
        "pre_total": 1503,
        "post_negligent": 1120,
        "post_total": 1270
    }
}

# Display the observed proportions
summary = []

for season, values in data.items():
    p_pre = values["pre_negligent"] / values["pre_total"]
    p_post = values["post_negligent"] / values["post_total"]

    summary.append({
        "Season": season,
        "Pre-2017 proportion": p_pre,
        "Post-2017 proportion": p_post,
        "Change (percentage points)": (p_post - p_pre) * 100
    })

summary_df = pd.DataFrame(summary)
summary_df

,Season,Pre-2017 proportion,Post-2017 proportion,Change (percentage points)
0,Critical,0.557487,0.474074,-8.341309
1,Non-critical,0.826347,0.881890,5.554246


## 2. Two-proportion Z-test

For each season, the test compares the proportion of negligent fires before and after 2017.

**Step 1 — Calculate the sample proportions**

\[
\\hat p_{pre}=\\frac{x_{pre}}{n_{pre}}
\\qquad
\\hat p_{post}=\\frac{x_{post}}{n_{post}}
\]

**Step 2 — Calculate the pooled proportion**

Under the null hypothesis, both periods have the same population proportion:

\[
\\hat p=\\frac{x_{pre}+x_{post}}{n_{pre}+n_{post}}
\]

**Step 3 — Calculate the standard error**

\[
SE=\\sqrt{\\hat p(1-\\hat p)
\\left(\\frac{1}{n_{pre}}+\\frac{1}{n_{post}}\\right)}
\]

**Step 4 — Calculate the Z-statistic**

\[
Z=\\frac{\\hat p_{post}-\\hat p_{pre}}{SE}
\]

The two-sided p-value is:

\[
p=2\\left[1-\\Phi(|Z|)\\right]
\]

where \\(\\Phi\\) is the cumulative distribution function of the standard normal distribution.

In [2]:
def two_proportion_z_test(x_pre, n_pre, x_post, n_post):
    """
    Two-sided two-proportion Z-test.
    x = number of negligent fires (successes)
    n = total number of human-caused fires
    """

    # Sample proportions
    p_pre = x_pre / n_pre
    p_post = x_post / n_post

    # Pooled proportion under H0: p_pre = p_post
    p_pooled = (x_pre + x_post) / (n_pre + n_post)

    # Pooled standard error
    standard_error = (
        p_pooled * (1 - p_pooled) *
        (1 / n_pre + 1 / n_post)
    ) ** 0.5

    # Z-statistic: post-2017 minus pre-2017
    z_stat = (p_post - p_pre) / standard_error

    # Two-sided p-value
    p_value = 2 * norm.sf(abs(z_stat))

    return {
        "Pre-2017 proportion": p_pre,
        "Post-2017 proportion": p_post,
        "Pooled proportion": p_pooled,
        "Standard error": standard_error,
        "Z-statistic": z_stat,
        "p-value": p_value
    }


# Run the test separately for each season
results = {}

for season, values in data.items():
    results[season] = two_proportion_z_test(
        x_pre=values["pre_negligent"],
        n_pre=values["pre_total"],
        x_post=values["post_negligent"],
        n_post=values["post_total"]
    )

results_df = pd.DataFrame(results).T
results_df

,Pre-2017 proportion,Post-2017 proportion,Pooled proportion,Standard error,Z-statistic,p-value
Critical,0.557487,0.474074,0.530432,0.017419,-4.788731,0.000002
Non-critical,0.826347,0.881890,0.851785,0.013543,4.101290,0.000041


## 3. Assumptions

- **Independence:** fires are treated as independent observations.
- **Large sample condition:** the numbers of negligent and non-negligent fires in each group are sufficiently large. The smallest count of successes or failures in the four groups is 150, which is above the usual minimum of 10.

## 4. Decision and interpretation

In [3]:
alpha = 0.05

for season, result in results.items():
    print(f"--- {season} period ---")
    print(f"Z-statistic: {result['Z-statistic']:.4f}")
    print(f"p-value: {result['p-value']:.3e}")

    if result["p-value"] < alpha:
        print("Decision: Reject H0.")
        print("There is evidence that the proportion changed after 2017.")
    else:
        print("Decision: Fail to reject H0.")
        print("There is not enough evidence that the proportion changed.")

    print()

--- Critical period ---
Z-statistic: -4.7887
p-value: 1.678e-06
Decision: Reject H0.
There is evidence that the proportion changed after 2017.

--- Non-critical period ---
Z-statistic: 4.1013
p-value: 4.109e-05
Decision: Reject H0.
There is evidence that the proportion changed after 2017.



## 5. Conclusion

The proportion of negligent fires changed after 2017, but in different directions across the two seasons.

- **Critical summer period:** the proportion decreased from 55.7% to 47.4%.
- **Non-critical months:** the proportion increased from 82.6% to 88.2%.

The two separate Z-tests show that the proportion changed significantly within each season. However, they do **not** directly test whether the size of the change differs significantly between seasons.

To test that final comparison directly, a logistic regression including a `Period * Season_Risk_Period` interaction term could be used.